In [0]:
from pyspark.sql import functions as F

# Read pipeline tables
bronze_df = spark.table("workspace.movies_dev.bronze_movies")
silver_df = spark.table("workspace.movies_dev.silver_movies")
quarantine_df = spark.table("workspace.movies_dev.quarantine_movies")

gold_industry = spark.table("workspace.movies_dev.gold_industry_analytics")
gold_studio = spark.table("workspace.movies_dev.gold_studio_analytics")
gold_top_rated = spark.table("workspace.movies_dev.gold_top_rated_movies")
gold_yearly = spark.table("workspace.movies_dev.gold_yearly_analytics")

# Record counts
bronze_count = bronze_df.count()
silver_count = silver_df.count()
quarantine_count = quarantine_df.count()

print("Bronze records:", bronze_count)
print("Silver records:", silver_count)
print("Quarantined records:", quarantine_count)

# Validation 1: Silver count should not exceed Bronze count
assert silver_count <= bronze_count, "Silver count exceeds Bronze count"

# Validation 2: No invalid IMDb ratings in Silver
invalid_ratings = silver_df.filter(
    F.col("imdb_rating").isNotNull() &
    ~F.expr("try_cast(imdb_rating as double)").between(0, 10)
).count()

assert invalid_ratings == 0, "Invalid IMDb ratings found in Silver"

# Validation 3: No duplicate movie keys in Silver
duplicate_count = (
    silver_df.groupBy("title", "release_year", "industry")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_count == 0, "Duplicate records found in Silver"

# Validation 4: Quarantine records must have a reason
missing_reason_count = quarantine_df.filter(
    F.col("quarantine_reason").isNull() |
    (F.trim(F.col("quarantine_reason")) == "")
).count()

assert missing_reason_count == 0, "Quarantine records without a reason found"

# Validation 5: Gold tables should not be empty when Silver has data
if silver_count > 0:
    assert gold_industry.count() > 0, "Industry Gold table is empty"
    assert gold_studio.count() > 0, "Studio Gold table is empty"
    assert gold_yearly.count() > 0, "Yearly Gold table is empty"

print("All quality checks passed successfully!")